# Taller 1: Econometría

- **Profesor:** Francisco Alfaro Medina
- **Ayudantes:** Krischnna Cortez y Karen Rojas


### Instrucciones

- Dispone de **60 minutos** para completar los **100 puntos** del taller.
- Cuide la presentación y redacción de sus respuestas.
- Puede utilizar su computador y los apuntes de clase y ayudantía.
- Debe entregar un archivo **PDF** y un **R script** (extensión `.R`).

> ⚠️ **Importante para Colab:** Este notebook usa un kernel de R. Si abre este archivo en Google Colab, seleccione **Runtime → Change runtime type → R** antes de ejecutar cualquier celda.

---
# Sección 1: Datos Aleatorios *(30 puntos)*

En esta sección trabajaremos con un dataset **simulado** de 50 alumnos de la USM. El dataset contiene las siguientes variables:

| Variable | Descripción |
|---|---|
| `hrs_sueno` | Horas de sueño promedio en el último mes |
| `profesor_part` | Si recibió ayuda de profesor particular (0/1) |
| `media_sem_pasado` | Promedio de notas del semestre anterior |
| `tiempo_est` | Horas de estudio dedicadas |
| `asistencia` | Porcentaje de asistencia a clases |
| `nivel_socioec` | Nivel socioeconómico (1 al 5) |
| `notas` | **Variable dependiente** — nota del alumno |

## Pregunta 1.1 — Generar el dataset *(6 pts.)*

Antes de ejecutar el código, **cambie la semilla** según la primera letra de su apellido:

| A–E | F–J | K–O | P–T | U–Z |
|:---:|:---:|:---:|:---:|:---:|
| 123 | 456 | 789 | 101112 | 131415 |

Reemplace el valor en `set.seed(...)` antes de continuar.

In [ ]:
# -------------------------------------------------------
# Pregunta 1.1: Generar el dataframe "datos"
# Cambie la semilla según la primera letra de su apellido
# A-E: 123 | F-J: 456 | K-O: 789 | P-T: 101112 | U-Z: 131415
# -------------------------------------------------------

set.seed(789)  # <-- CAMBIE ESTE VALOR SEGÚN SU APELLIDO

datos <- data.frame(
  hrs_sueno        = round(runif(50, min = 5,  max = 10),  1),
  profesor_part    = sample(c(0, 1), 50, replace = TRUE),
  media_sem_pasado = round(runif(50, min = 60, max = 100), 1),
  tiempo_est       = round(runif(50, min = 1,  max = 8),   1),
  asistencia       = round(runif(50, min = 60, max = 100), 1),
  nivel_socioec    = sample(1:5, 50, replace = TRUE)
)

# Calcular notas con ponderaciones definidas
datos$notas <- 30 +
  datos$hrs_sueno        * 1.5  +
  datos$profesor_part    * 3    +
  datos$media_sem_pasado * 0.2  +
  datos$tiempo_est       * 2    +
  datos$asistencia       * 0.15 +
  datos$nivel_socioec    * 2    +
  rnorm(50, mean = 0, sd = 5)

# Asegurar rango entre 20 y 100
datos$notas <- pmax(pmin(datos$notas, 100), 20)

# Vista rápida del dataset
cat("Dimensiones del dataset:", nrow(datos), "filas x", ncol(datos), "columnas\n")
head(datos)

Dimensiones del dataset: 50 filas x 7 columnas


,hrs_sueno,profesor_part,media_sem_pasado,tiempo_est,asistencia,nivel_socioec,notas
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,8.5,0,79.2,3.8,71.7,2,78.60754
2,5.5,0,72.6,6.1,63.7,5,80.41091
3,5.1,1,62.6,7.5,73.8,4,93.97003
4,8.0,1,95.6,4.2,68.6,3,84.18621
5,7.5,1,91.5,6.5,88.8,2,90.67901
6,5.1,0,66.5,1.2,97.1,3,70.89307


## Pregunta 1.2 — Redondear notas *(2 pts.)*

Redondee la variable `notas` a **1 decimal**.

In [ ]:
# Pregunta 1.2: Redondear notas a 1 decimal
set.seed(789)  # <-- CAMBIE ESTE VALOR SEGÚN SU APELLIDO

datos <- data.frame(
  hrs_sueno        = round(runif(50, min = 5,  max = 10),  1),
  profesor_part    = sample(c(0, 1), 50, replace = TRUE),
  media_sem_pasado = round(runif(50, min = 60, max = 100), 1),
  tiempo_est       = round(runif(50, min = 1,  max = 8),   1),
  asistencia       = round(runif(50, min = 60, max = 100), 1),
  nivel_socioec    = sample(1:5, 50, replace = TRUE)
)

datos$notas <- 30 +
  datos$hrs_sueno        * 1.5  +
  datos$profesor_part    * 3    +
  datos$media_sem_pasado * 0.2  +
  datos$tiempo_est       * 2    +
  datos$asistencia       * 0.15 +
  datos$nivel_socioec    * 2    +
  rnorm(50, mean = 0, sd = 5)

datos$notas <- round(pmax(pmin(datos$notas, 100), 20),1)

# Vista rápida del dataset
cat("Dimensiones del dataset:", nrow(datos), "filas x", ncol(datos), "columnas\n")
head(datos)

Dimensiones del dataset: 50 filas x 7 columnas


,hrs_sueno,profesor_part,media_sem_pasado,tiempo_est,asistencia,nivel_socioec,notas
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,8.5,0,79.2,3.8,71.7,2,78.6
2,5.5,0,72.6,6.1,63.7,5,80.4
3,5.1,1,62.6,7.5,73.8,4,94.0
4,8.0,1,95.6,4.2,68.6,3,84.2
5,7.5,1,91.5,6.5,88.8,2,90.7
6,5.1,0,66.5,1.2,97.1,3,70.9


## Pregunta 1.3 — Estimadores β via álgebra matricial *(8 pts.)*

Calcule los estimadores MCO **manualmente**, usando la fórmula matricial:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Sin usar** la función `lm()` de R.

In [ ]:
# Pregunta 1.3: Estimadores beta via álgebra matricial (sin lm)


Warning message in mean.default(datos$X):
“argument is not numeric or logical: returning NA”
Warning message in mean.default(datos$Y):
“argument is not numeric or logical: returning NA”



=== Parte (a): Estimadores MCO ===
X̄ = NA
Ȳ = NA


ERROR: Error in mutate(datos, desv_X = X - X_bar, desv_Y = Y - Y_bar, prod = desv_X * : could not find function "mutate"


## Pregunta 1.4 — Modelo con `lm()` e interpretación *(8 pts.)*

Genere el modelo de regresión múltiple usando la función `lm()` y obtenga el resumen con `summary()`.  
Luego, **interprete cada coeficiente** en el espacio indicado.

> 💡 **Tip:** Los β de `lm()` deben coincidir con los calculados manualmente en la pregunta anterior.

In [ ]:
# Pregunta 1.4: Modelo con lm() y summary
modelo <- lm(notas ~ hrs_sueno + profesor_part + media_sem_pasado + tiempo_est + asistencia + nivel_socioec, data = datos)
cat("\n--- Verificación con lm() ---\n")
print(coef(modelo))

# Obtener el resumen del modelo
summary(modelo)

ERROR: Error in eval(mf, parent.frame()): object 'datos' not found


## Pregunta 1.5 — Relación entre betas y ponderaciones del código *(6 pts.)*

Analice la relación entre los coeficientes estimados (β̂) y las ponderaciones reales usadas en el código del Anexo para generar las notas.



In [ ]:
# Pregunta 1.5: Comparación entre betas estimados y ponderaciones reales


# Sección 2: Wooldridge *(70 puntos)*

Para esta sección utilizaremos el paquete `wooldridge`, que contiene bases de datos clásicas de econometría.

In [ ]:
# Instalar y cargar el paquete wooldridge (solo necesario la primera vez en Colab)
if (!require(wooldridge)) install.packages("wooldridge")
library(wooldridge)

Loading required package: wooldridge

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘wooldridge’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



---
## 2A. Base `wage1` *(30 puntos)*

El modelo de regresión poblacional a estimar es:

$$wage = \beta_0 + \beta_1\, educ + \beta_2\, exper + \beta_3\, tenure + u$$

Donde:
- `wage` = salario por hora (USD)
- `educ` = años de escolaridad
- `exper` = años de experiencia laboral
- `tenure` = años en el trabajo actual

In [ ]:
# Cargar y limpiar la base wage1
data("wage1")
wage1<-na.omit(wage1)

### Pregunta 2A.1 — Estimadores MCO e interpretación *(8 pts.)*

Estime el modelo completo e interprete los resultados.

In [ ]:
# Pregunta 2A.1: Modelo de regresión múltiple con wage1
modelo1 <- lm(wage ~ educ + exper + tenure, data = wage1)
summary(modelo1)


Call:
lm(formula = wage ~ educ + exper + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.6068 -1.7747 -0.6279  1.1969 14.6536 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.87273    0.72896  -3.941 9.22e-05 ***
educ         0.59897    0.05128  11.679  < 2e-16 ***
exper        0.02234    0.01206   1.853   0.0645 .  
tenure       0.16927    0.02164   7.820 2.93e-14 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.084 on 522 degrees of freedom
Multiple R-squared:  0.3064,	Adjusted R-squared:  0.3024 
F-statistic: 76.87 on 3 and 522 DF,  p-value: < 2.2e-16


### Pregunta 2A.2 — ¿Los signos son los esperados? *(10 pts.)*

Antes de ver los resultados, reflexione: ¿qué signo debería tener cada coeficiente económicamente?

In [ ]:
# Pregunta 2A.2: Revisar signos de los coeficientes


Los estimadores si presentan los signos esperados en el cual vemos que es consistente, ya que, todos son mayor a cero debido a mayor educación, experiencia y antigüedad deberían aumentar el salario.
Viendo cada coeficiente economicamente, todos deberian ser mayor a cero para poder asi aumentar el salario

### Pregunta 2A.3 — Comparar R² y R² ajustado *(12 pts.)*

Estime un segundo modelo usando solo `educ` y `tenure`, y compare el ajuste con el modelo completo.

In [ ]:
# Pregunta 2A.3: Modelo reducido (sin exper)
modelo2 <- lm(wage ~ educ + tenure, data = wage1)
summary(modelo2)

summary(modelo2)$r.squared
summary(modelo1)$r.squared

summary(modelo2)$adj.r.squared
summary(modelo1)$adj.r.squared


Call:
lm(formula = wage ~ educ + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-8.1438 -1.7288 -0.6372  1.2575 14.7482 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.22162    0.64015   -3.47 0.000563 ***
educ         0.56914    0.04881   11.66  < 2e-16 ***
tenure       0.18958    0.01871   10.13  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.092 on 523 degrees of freedom
Multiple R-squared:  0.3019,	Adjusted R-squared:  0.2992 
F-statistic: 113.1 on 2 and 523 DF,  p-value: < 2.2e-16


[1] 0.301861

[1] 0.3064224

[1] 0.2991912

[1] 0.3024364

El modelo completo presenta un R^2 de 0.3064 y un R^2 ajustado de 0.3024, mientras que el modelo reducido tiene un R^2 de 0.3019 y un R^2 ajustado de 0.2992. Dado que ambos indicadores son mayores en el modelo completo, se concluye que este presenta un mejor ajuste y que la variable experiencia aporta capacidad explicativa, por lo que es conveniente mantenerla en el modelo.

---
## 2B. Base `attend` *(40 puntos)*

En esta sección analizamos los determinantes del rendimiento en el examen final de un curso universitario.

### Pregunta 2B.1 — Cargar la base `attend` *(4 pts.)*

In [ ]:
# Pregunta 2B.1: Cargar base attend
data("attend")
attend <- na.omit(attend)

### Pregunta 2B.2 — Selección de variables *(4 pts.)*

Subseleccione las siguientes variables:

| Variable | Descripción |
|---|---|
| `attend` | Clases asistidas de un total de 32 |
| `termGPA` | Promedio de notas durante el período |
| `priGPA` | Promedio acumulado antes del período |
| `ACT` | Puntaje en el examen ACT |
| `final` | **Variable dependiente** — puntaje del examen final |
| `hwrte` | Porcentaje de tareas entregadas |
| `frosh` | =1 si es estudiante de primer año |
| `soph` | =1 si es estudiante de segundo año |

In [ ]:
# Pregunta 2B.2: Subselección de variables

datos_attend <- attend[, c("attend", "termGPA", "priGPA", "ACT", "final", "hwrte", "frosh", "soph")]

# Ver primeras filas
head(datos_attend)

,attend,termGPA,priGPA,ACT,final,hwrte,frosh,soph
,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<int>
1,27,3.19,2.64,23,28,100.0,0,1
2,22,2.73,3.52,25,26,87.5,0,0
3,30,3.00,2.46,24,30,87.5,0,0
4,31,2.04,2.61,20,27,100.0,0,1
5,32,3.68,3.32,23,34,100.0,0,1
6,29,3.23,2.93,26,25,100.0,0,1


### Pregunta 2B.3 — Modelo de regresión múltiple completo *(10 pts.)*

Estime un modelo donde la variable dependiente es `final` y las independientes son todas las demás variables del subconjunto.

In [ ]:
# Pregunta 2B.3: Modelo completo con attend_new
modelo_final <- lm(final ~ attend + termGPA + priGPA + ACT + hwrte + frosh + soph, data = datos_attend)

summary(modelo_final)


Call:
lm(formula = final ~ attend + termGPA + priGPA + ACT + hwrte + 
    frosh + soph, data = datos_attend)

Residuals:
     Min       1Q   Median       3Q      Max 
-15.9573  -2.5968  -0.0042   2.6582  11.0815 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 14.27176    1.45667   9.798  < 2e-16 ***
attend      -0.06956    0.04105  -1.694   0.0907 .  
termGPA      3.54151    0.31442  11.264  < 2e-16 ***
priGPA      -0.04149    0.40302  -0.103   0.9180    
ACT          0.27313    0.05031   5.429 7.94e-08 ***
hwrte       -0.01394    0.01043  -1.337   0.1817    
frosh       -0.61310    0.47672  -1.286   0.1989    
soph        -0.82251    0.39459  -2.084   0.0375 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.867 on 666 degrees of freedom
Multiple R-squared:  0.3373,	Adjusted R-squared:  0.3303 
F-statistic: 48.42 on 7 and 666 DF,  p-value: < 2.2e-16


### Pregunta 2B.4 — Bondad de ajuste *(6 pts.)*

Interprete el **R²** y el **R² ajustado** del modelo anterior.

In [ ]:
# Pregunta 2B.4: Extraer métricas de bondad de ajuste
summary(modelo_final)$r.squared
summary(modelo_final)$adj.r.squared

[1] 0.3372808

[1] 0.3303152

### Pregunta 2B.5 — Modelo reducido (excluir variables no significativas) *(10 pts.)*

Genere un nuevo modelo excluyendo las variables con **p-value > 0.05** en el modelo anterior.

In [ ]:
# Identificar variables significativas (p-value <= 0.05)

In [ ]:
# Pregunta 2B.5: Modelo reducido (solo variables significativas)
# Variables significativas identificadas: termGPA, ACT, soph
# (ajuste según los resultados de su modelo)

### Pregunta 2B.6 — Comparación de modelos *(6 pts.)*

Compare el modelo completo y el modelo reducido en términos de **R² ajustado**.

In [ ]:
# Pregunta 2B.6: Comparación final de modelos